# C02 财务 + 衍生 + 宏观因子（在线版）

本节全部使用 `rqdatac` 在线接口构建 `factor_panel`。


In [ ]:
START_DATE = "2022-01-01"
END_DATE = "2024-12-31"
UNIVERSE = ["000001.XSHE", "000002.XSHE", "600000.XSHG", "601318.XSHG"]

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import rqdatac

# 方式1（推荐课堂演示）：把教育版 license 直接粘贴到 PASSWD
PASSWD = "在这里粘贴你的教育版license"

# 方式2（不想明文写在 notebook）：注释掉上面一行，改用环境变量
# import os
# PASSWD = os.getenv("RQDATAC_LICENSE", "")

if PASSWD and ("在这里粘贴" not in PASSWD):
    rqdatac.init('license', PASSWD)
    print("rqdatac 初始化成功")
else:
    print("请先填写 PASSWD，再运行本单元")


## 1) 财务数据接口示例
- `get_pit_financials_ex`
- `get_factor`


In [ ]:
pit_fin = rqdatac.get_pit_financials_ex(
    fields=["revenue", "net_profit"],
    start_quarter="2021q1",
    end_quarter="2024q3",
    order_book_ids=UNIVERSE,
    statements="latest",
)

pit_fin.head()


In [ ]:
fin_factor = rqdatac.get_factor(
    UNIVERSE,
    ["revenue_ttm_0", "net_profit_ttm_0", "pe_ratio_ttm"],
    start_date=START_DATE,
    end_date=END_DATE,
)

fin_factor.head()


## 2) 衍生因子 + 宏观序列
宏观接口在不同版本字段可能有差异，这里用 `try/except` 保持课堂可继续。


In [ ]:
tech_factor = rqdatac.get_factor(
    UNIVERSE,
    ["WorldQuant_alpha010", "rsi_14", "macd_dif"],
    start_date=START_DATE,
    end_date=END_DATE,
)

try:
    # 宏观数据接口字段依账号权限可能略有不同
    rr = rqdatac.econ.get_reserve_ratio()
    macro = rr[(rr.index >= START_DATE) & (rr.index <= END_DATE)].copy()
except Exception as e:
    print("宏观接口暂不可用，先保留空表：", e)
    macro = pd.DataFrame(columns=["macro_value"])

tech_factor.head(), macro.head()


## 3) 组装月频 factor_panel


In [ ]:
close = rqdatac.get_price(
    UNIVERSE,
    start_date=START_DATE,
    end_date=END_DATE,
    fields="close",
    adjust_type="post",
    expect_df=False,
)

close = close.droplevel(0, axis=1) if isinstance(close.columns, pd.MultiIndex) else close
mom20 = close.pct_change(20)
vol20 = close.pct_change().rolling(20).std()

month_ends = pd.DatetimeIndex(close.resample("ME").last().index)
rows = []
for d in month_ends:
    for asset in UNIVERSE:
        rows.append({
            "date": d,
            "order_book_id": asset,
            "mom20": mom20.loc[d, asset] if d in mom20.index else np.nan,
            "vol20": vol20.loc[d, asset] if d in vol20.index else np.nan,
        })

factor_panel = pd.DataFrame(rows)
for col in ["mom20", "vol20"]:
    factor_panel[col] = factor_panel.groupby("date")[col].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))

factor_panel.head(12)


## 小结
本节完全在线拉取财务与衍生数据，构建了月频 `factor_panel`。
